# Разработка алгоритмов машинного обучения для прогнозирования показателей социально-экономического развития

**Тема ВКР:** прогнозирование ВРП на душу населения регионов РФ с использованием
классических и современных методов машинного обучения.

**Объект исследования:** социально-экономическое развитие 85 субъектов
Российской Федерации.

**Предмет исследования:** методы машинного обучения и экономико-математического
моделирования для прогнозирования ВРП на душу населения и связанных
социально-экономических показателей.

**Целевая переменная:** ВРП на душу населения по регионам РФ.
В работе рассматриваются обе формы показателя:
* **номинальный ВРП на душу населения** — в текущих ценах;
* **реальный ВРП на душу населения в ценах 2015 года** — очищенный от влияния
  инфляции.

**Период данных:** 2001–2023 годы.

**Источники:** Росстат / ЕМИСС, World Bank, Банк России.


## 1. Введение

### 1.1. Актуальность темы

Прогнозирование социально-экономического развития регионов является одной из
ключевых задач государственного управления. Качественные региональные прогнозы
позволяют:

* **формировать обоснованные бюджеты** на 3-летний период (БК РФ);
* **планировать инвестиционные программы** Минэкономразвития и национальных
  проектов;
* **выявлять регионы риска** на ранних стадиях экономических шоков;
* **оценивать эффективность мер региональной политики**;
* **мониторить территориальное неравенство** (коэффициент Джини, доля 10
  богатейших / 10 беднейших субъектов).

Применение **алгоритмов машинного обучения** к региональным панельным данным —
это новое направление, дополняющее классические методы (Panel Fixed Effects,
ARIMA), за счёт способности:
* улавливать нелинейные взаимодействия факторов;
* работать с разреженными многомерными данными;
* давать интерпретацию через SHAP-значения;
* строить сценарные прогнозы в режиме «what-if».

### 1.2. Цель и задачи

**Цель:** разработать комплекс алгоритмов машинного обучения для прогнозирования
ВРП на душу населения регионов РФ и реализовать прототип аналитической системы
со сценарным прогнозом и картой регионов.

**Задачи:**
1. Сформировать панельный датасет по 85 регионам РФ за 2001–2023 гг.
2. Реализовать корректный расчёт реального ВРП в ценах 2015 г.
3. Сконструировать признаковое пространство без утечек данных.
4. Сравнить классические и современные ML-методы прогнозирования.
5. Реализовать walk-forward валидацию для honest out-of-sample оценки.
6. Подобрать гиперпараметры через TimeSeriesSplit.
7. Получить SHAP-интерпретацию для лучшей модели.
8. Построить сценарные прогнозы (baseline / opt / pess) на горизонте 1–5 лет.
9. Реализовать интерактивную карту регионов.
10. Разработать прототип Streamlit-интерфейса.

### 1.3. Гипотеза исследования

**Основная гипотеза:** современные ML-алгоритмы (gradient boosting, random
forest) дают значимое улучшение качества прогноза ВРП на душу населения
по сравнению с классическими линейными моделями за счёт улавливания
нелинейных эффектов; при этом инерционные бенчмарки (MeanGrowth) остаются
сильными конкурентами на коротком 1-летнем горизонте — это типичная картина
для коротких макроэкономических рядов.

**Дополнительные гипотезы:**
* лаги целевой переменной (`log(ВРП)`) являются доминирующим предиктором;
* макрофакторы (нефть, ставка, инфляция, курс) — устойчивые драйверы;
* SHAP-вклад макрофакторов специфичен по типам регионов (сырьевые vs
  индустриальные vs сервисные).


## 2. Теоретико-экономическое обоснование

### 2.1. ВРП и ВРП на душу населения

**Валовой региональный продукт (ВРП)** — стоимостная оценка произведённых в
регионе товаров и услуг за год. Аналог ВВП на уровне субъекта федерации.
Рассчитывается Росстатом производственным методом.

**ВРП на душу населения** = ВРП ÷ среднегодовая численность населения.
Универсальный показатель уровня экономического развития, сравнимый между
регионами разного размера.

### 2.2. Номинальный vs реальный ВРП

| Характеристика | Номинальный | Реальный |
|----------------|-------------|----------|
| В каких ценах | текущих | базового года |
| Что отражает | объём + цены | только физический объём |
| Влияние инфляции | да | очищен |
| Сопоставимость во времени | низкая | высокая |
| Источник в стат. отчётности | Росстат, Y477110006 | расчётный |

**Зачем нужен реальный ВРП:** при инфляции 8–15%/год номинальный рост в 10%
может означать реальное падение. Только в реальных ценах видна
**физическая динамика производства**.

### 2.3. Выбор базового года — 2015

Базовый год выбран:
1. **все 85 регионов** имеют данные за 2015 (включая Крым и Севастополь, с 2014);
2. **2015 — стандартная база** международных индексов (МВФ, World Bank);
3. **8 лет до конца периода** — небольшая накопленная ошибка дефлятора;
4. соответствие методическим стандартам Росстата.

### 2.4. Дефлятор

Используется **официальный Индекс физического объёма ВРП Росстата (Y477110109)** —
% к предыдущему году в постоянных ценах.

Рекурсивный расчёт:
$$\text{real\_factor}(i, 2015) = 1.0$$
$$\text{real\_factor}(i, t) = \text{real\_factor}(i, t-1) \cdot \frac{\text{ИФО}(i, t)}{100}, \quad t > 2015$$
$$\text{real\_factor}(i, t) = \text{real\_factor}(i, t+1) \div \frac{\text{ИФО}(i, t+1)}{100}, \quad t < 2015$$

Реальный ВРП на душу:
$$\text{ВРП}_{real, i, t} = \text{ВРП}_{nom, i, 2015} \cdot \text{real\_factor}(i, t)$$

### 2.5. Факторы, влияющие на ВРП на душу населения

Базовая теоретическая модель — **производственная функция Кобба–Дугласа**
расширенная компонентами совокупного спроса:
$$Y = A \cdot K^{\alpha} \cdot L^{1-\alpha} = C + I + G + NX$$

| Группа фактора | Признак | Знак | Обоснование |
|----------------|---------|------|--------------|
| Капитал (K) | Инвестиции в осн. капитал на душу | + | Прямой драйвер выпуска |
| Капитал (K) | Объём инвестиций региона | + | Накопление капитала |
| Труд (L) | Численность рабочей силы | + | Объём фактора труда |
| Труд (L) | Безработица | − | Снижает реализацию факторов |
| Труд (L) | Уровень занятости | + | Прямой ресурс |
| Чел. капитал | Зарплата | + | Прокси производительности |
| Доходы (C) | Среднедушевые доходы | + | Драйвер потребления |
| Промышленность | Индекс пром. производства | + | Реальный выпуск |
| Промышленность | Индекс добычи | + | Сырьевая компонента |
| Торговля (C) | Розничный товарооборот | + | Внутренний спрос |
| Внешний сектор (NX) | Экспорт | + | Внешний спрос |
| Внешний сектор (NX) | Импорт | + | Прокси инвест. активности |
| Стройка (I) | Объём строительных работ | + | Инвест. компонента |
| Демография | Миграционный прирост | + | Приток трудовых ресурсов |
| Демография | Прод. жизни | + | Человеческий капитал |
| Цены | ИПЦ | ? | Двойственно: ↑ инфляция → ↑ ном. ВРП, но ↓ реальный |
| Инфраструктура | Доступ к врачам | − | Чем меньше нас./врача, тем лучше |

**Лаги и временная структура:**
* эффект **инвестиций** проявляется с лагом 1–3 года (gestation lag);
* **демографические факторы** действуют через длинные лаги (5–10 лет);
* **макрошоки** (нефть, ставка) — частично мгновенно, частично с лагом 1 год;
* поэтому в модели мы используем **только лаги** факторов (защита от утечек +
  экономически обоснованная задержка эффекта).


## 3. Описание данных

* **Объект:** 85 регионов РФ.
* **Период:** 2001–2023 гг.
* **Источники:**
  * **Росстат / ЕМИСС** — региональные индикаторы (`data_regions_collection_*.parquet`);
  * **World Bank** — рост ВВП РФ, инфляция, безработица (`macro_external.csv`);
  * **Банк России** — ключевая ставка ЦБ;
  * **World Bank Commodity Markets** — цены на нефть Brent.

* **Сырых наблюдений** в long-формате: ~1.5 млн.
* **Уникальных индикаторов:** > 1200.
* **После фильтрации** (object_level == "Регион", 2001–2023, без агрегатов): ~1.5 млн.

После pivot в wide-формат: 1939 строк (85 регионов × 23 года) × 913 колонок.
После удаления разреженных (>60% пропусков): ~500 признаков.

### 3.1. Каталог переменных


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks_vkr' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src_vkr.config import (
    DATA_PROC, RESULTS, TARGET_NOM, GRP_REAL_PC, MACRO_COLS,
    TRAIN_END, TEST_START, TEST_END, BASE_YEAR, EXPERT_INDICATORS,
)
# Единый источник интерактивных фигур (тот же модуль, что в Streamlit-app).
from src_vkr import viz_common as vc

catalog = pd.read_csv(DATA_PROC / 'feature_catalog.csv')
print(f'Всего индикаторов в каталоге: {len(catalog)}')
catalog.head(15)

Всего индикаторов в каталоге: 911


,indicator_code,indicator_name,unit_canonical
0,Y477030027,Инвестиции в жилые здания и помещения,Млн руб
1,Y477190005,Индексы цен (тарифов) на товары и услуги: Инде...,"Декабрь к декабрю предыдущего года, в процентах"
2,Y477190004,Индексы цен (тарифов) на товары и услуги: Инде...,"Декабрь к декабрю предыдущего года, в процентах"
3,Y477190017,Средние цены на рынке жилья: Стоимость 1 кв. м...,Тыс. руб
4,Y477190016,Средние цены на рынке жилья: Средние цены на п...,"На конец года, рублей за квадратный метр общей..."
5,Y477190015,Средние цены на рынке жилья: Средние цены на в...,"На конец года, рублей за квадратный метр общей..."
6,Y477100008,Уровень безработицы населения,В процентах
7,Y477100014,Численность безработных,Тыс чел
8,Y477190008,Индексы цен на рынке жилья: Индексы цен на пер...,"На конец года, в процентах к концу предыдущего..."
9,Y477190007,Индексы цен на рынке жилья: Индексы цен на вто...,"На конец года, в процентах к концу предыдущего..."


In [16]:
# Экспертный пул кандидатных индикаторов с экономической интерпретацией
ind_df = pd.DataFrame([
    {'code': k, 'name': v[0], 'group': v[1], 'expected_sign': v[2], 'rationale': v[3]}
    for k, v in EXPERT_INDICATORS.items()
])
ind_df

,code,name,group,expected_sign,rationale
0,Y477110108,Инвестиции в осн. капитал на душу нас.,capital,+,Прямой драйвер производственной функции — рост...
1,Y477140022,"Инвестиции в осн. капитал, млн руб.",capital,+,Региональный объём инвестиций — фактор накопле...
2,Y477140025,Индекс физ. объёма инвестиций,capital,+,Динамика инвестиций в сопоставимых ценах.
3,Y477110378,"Среднемесячная номин. з/п, руб.",labor,+,Прокси производительности труда и платежеспосо...
4,Y477100008,"Уровень безработицы (МОТ), %",labor,-,Рост безработицы снижает выпуск и доходы.
5,Y477100010,"Уровень занятости, %",labor,+,Прямой ресурс производства.
6,Y477100012,"Уровень участия в рабочей силе, %",labor,+,Активная часть населения.
7,Y477100014,"Численность безработных, тыс чел.",labor,-,Альтернативная оценка безработицы по численности.
8,Y477100023,"Численность рабочей силы, тыс чел.",labor,+,Объём трудовых ресурсов.
9,Y477110374,"Среднедушевые денежные доходы, руб/мес",income,+,Прямой драйвер потребления C.


## 4. Предобработка данных

Этапы:
1. загрузка long-формата;
2. фильтрация регионов и периода;
3. замена спецзначений (-99999999) на NaN;
4. дедупликация по свежести источника;
5. нормализация единиц (млн руб., тыс чел., руб.);
6. pivot в wide;
7. подключение макрофакторов;
8. расчёт реального ВРП через дефлятор;
9. удаление колонок с покрытием < 40%.


In [17]:
# Загружаем уже выполненный препроцессинг
panel = pd.read_parquet(DATA_PROC / 'panel_preprocessed.parquet')
print('Shape:', panel.shape)
print('Regions:', panel['object_name'].nunique())
print('Years:', panel['year'].min(), '-', panel['year'].max())
panel[['object_name','year', TARGET_NOM, GRP_REAL_PC, 'grp_nom_growth', 'grp_real_growth']].head(15)

Shape: (1939, 509)
Regions: 85
Years: 2001 - 2023


,object_name,year,Y477110006,grp_real_pc_2015,grp_nom_growth,grp_real_growth
0,Алтайский край,2001,23509.0,115277.320008,NaN,NaN
1,Алтайский край,2002,27991.2,119657.858168,19.065890,3.800000
2,Алтайский край,2003,34295.8,128871.513247,22.523507,7.700000
3,Алтайский край,2004,44934.9,138150.262201,31.021583,7.200000
4,Алтайский край,2005,53812.4,142156.619805,19.756359,2.900000
5,Алтайский край,2006,69852.0,158788.944322,29.806513,11.700000
6,Алтайский край,2007,90759.9,175620.572420,29.931713,10.600000
7,Алтайский край,2008,106019.5,182645.395317,16.813152,4.000000
8,Алтайский край,2009,109088.7,173878.416342,2.894939,-4.800000
9,Алтайский край,2010,124955.8,180138.039330,14.545136,3.600000


**Проверка дефлятора:** реальный ВРП в 2015 г. должен равняться номинальному.

In [18]:
sub = panel[(panel['object_name']=='Алтайский край') & (panel['year'].between(2013, 2018))]
sub[['year', TARGET_NOM, GRP_REAL_PC, 'real_factor']].assign(
    diff_2015 = lambda d: np.where(d['year']==2015, d[TARGET_NOM]-d[GRP_REAL_PC], np.nan)
)

,year,Y477110006,grp_real_pc_2015,real_factor,diff_2015
12,2013,175666.0,204132.907480,0.976604,NaN
13,2014,189679.2,206456.840690,0.987722,NaN
14,2015,209023.2,209023.200000,1.000000,0.0
15,2016,230047.1,208711.508806,0.998509,NaN
16,2017,238056.6,212344.364612,1.015889,NaN
17,2018,256081.7,219150.961742,1.048453,NaN


## 5. Проверка признаков на наличие утечек данных

### 5.1. Что такое утечка
Утечка (data leakage) — это ситуация, когда модель получает в обучающем
наборе информацию, недоступную в момент прогнозирования. Для прогноза
ВРП на год t это означает:

| Что нельзя в признаке | Почему |
|------------------------|---------|
| Значение целевой переменной в году t | прямо равно ответу |
| Значение `ВРП_total(t)` | через деление на население = ВРП/чел. |
| Производные показатели текущего года | например, доходы бюджета (= налоги с ВРП) |
| Будущие значения признаков (year > t) | модель «подсматривает» |
| Признаки, публикуемые **после** t | например, ИФО за 2024 будет доступен только в 2025 |

### 5.2. Подход к удалению утечек

* Все экзогенные признаки (Y477...) попадают в признаковое пространство
  **только через лаги** (lag1, lag2, yoy_lag1).
* Целевая переменная — также только через лаги (target_lag1/2/3, roll3, growth).
* Макрофакторы — через лаги (_lag1).
* Для сценарного прогноза макрофакторы в году h задаются ВНЕШНЕ (сценарий),
  а не предсказываются моделью.

### 5.3. Аудит признаков


In [19]:
leak_audit = pd.read_csv(DATA_PROC / 'leak_audit.csv')
print(f'Кандидатных признаков: {len(leak_audit)}')
print(f'Подозрения на утечку: {leak_audit["suspect_leak"].sum()}')
leak_audit.head(20)

Кандидатных признаков: 130
Подозрения на утечку: 0


,feature,corr_with_target,has_lag_in_name,suspect_leak
0,nom_log_lag1,0.9961,True,False
1,nom_log_roll3_mean,0.9930,True,False
2,nom_log_lag2,0.9909,True,False
3,nom_log_lag3,0.9857,True,False
4,Y477110374_lag2,0.9025,True,False
5,Y477110374_lag1,0.8973,True,False
6,Y477110378_lag2,0.8864,True,False
7,Y477110378_lag1,0.8862,True,False
8,real_log_lag3,0.8530,True,False
9,real_log_lag2,0.8351,True,False


## 6. Формирование признаков

Все лаги и rolling-признаки строятся **внутри региона** (`groupby("object_name")`),
чтобы не было утечки между регионами.

Группы признаков:
1. **Лаги целевой переменной** (log-шкала): lag1, lag2, lag3.
2. **Скользящие** по лагам: roll3_mean, roll3_std.
3. **Темпы роста**: growth1 (1Y), growth3 (3Y).
4. **Лаги ключевых индикаторов** (Y477...): lag1, lag2, yoy_lag1.
5. **Лаги макрофакторов** (8 шт): _lag1.
6. **Временной тренд** (year_norm).
7. **Кластер региона** (экспертная типология).
8. **Федеральный округ** (one-hot).
9. **Относительная позиция** (лог-разница с медианой РФ).


In [20]:
df = pd.read_parquet(DATA_PROC / 'panel_features.parquet')
print('Shape after feature engineering:', df.shape)
print()
sel = ['nom_log_lag1','nom_log_lag2','nom_log_roll3_mean','nom_growth1','real_log_lag1',
       'real_growth1','year_norm','cluster_id']
df[['object_name','year'] + sel].dropna().head(10)

Shape after feature engineering: (1939, 642)



,object_name,year,nom_log_lag1,nom_log_lag2,nom_log_roll3_mean,nom_growth1,real_log_lag1,real_growth1,year_norm,cluster_id
2,Алтайский край,2003,10.239681,10.065181,10.152431,0.174500,11.692400,0.037295,0.090909,3
3,Алтайский край,2004,10.442807,10.239681,10.249223,0.203126,11.766579,0.074179,0.136364,3
4,Алтайский край,2005,10.712992,10.442807,10.465160,0.270185,11.836104,0.069526,0.181818,3
5,Алтайский край,2006,10.893278,10.712992,10.683026,0.180285,11.864692,0.028587,0.227273,3
6,Алтайский край,2007,11.154148,10.893278,10.920139,0.260871,11.975338,0.110646,0.272727,3
7,Алтайский край,2008,11.415984,11.154148,11.154470,0.261836,12.076087,0.100749,0.318182,3
8,Алтайский край,2009,11.571388,11.415984,11.380507,0.155404,12.115307,0.039220,0.363636,3
9,Алтайский край,2010,11.599926,11.571388,11.529099,0.028538,12.066117,-0.049190,0.409091,3
10,Алтайский край,2011,11.735723,11.599926,11.635679,0.135798,12.101484,0.035367,0.454545,3
11,Алтайский край,2012,11.834842,11.735723,11.723497,0.099118,12.146776,0.045292,0.500000,3


## 7. Отбор признаков

Этапы:
1. удаление константных / почти-константных (VarianceThreshold);
2. удаление дубликатов колонок;
3. удаление сильно коррелирующих пар (|ρ| > 0.92), из пары оставляем
   более экономически значимую / более важную по LightGBM gain;
4. ML-отбор: ранжирование по **0.5 · gain + 0.5 · permutation importance**
   на валидации 2018–2019 (без подсматривания в test 2020–2023);
5. форсированное включение 8 макрофакторов (для сценариев);
6. VIF-диагностика финального набора.


In [21]:
sel = pd.read_csv(DATA_PROC / 'selected_features.csv')['feature'].tolist()
imp = pd.read_csv(DATA_PROC / 'feature_importance.csv')
audit = pd.read_csv(DATA_PROC / 'feature_drop_audit.csv')
print(f'Финальный набор: {len(sel)} признаков\n')
for f in sel:
    print(f'  • {f}')

Финальный набор: 20 признаков

  • key_rate_avg_lag1
  • usd_rub_avg_lag1
  • gdp_growth_rf_wb_lag1
  • inflation_rf_wb_lag1
  • unemployment_rf_wb_lag1
  • industrial_production_rf_rosstat_lag1
  • oil_brent_price_avg_usd_lag1
  • nom_log_lag1
  • real_log_lag1
  • year_norm
  • Y477110378_lag1
  • Y477110108_lag1
  • cluster_id
  • Y477110341_lag2
  • fo_Северо-Кавказский ФО
  • Y477100014_lag1
  • Y477110301_lag2
  • Y477110378_yoy_lag1
  • Y477140022_lag1
  • Y477110374_yoy_lag1


In [22]:
# Какие признаки были удалены
audit.head(15)

,dropped,kept,r,reason
0,Y477110364_lag2,Y477110364_lag1,1.0000,duplicate column
1,Y477110125_lag2,Y477110125_lag1,1.0000,duplicate column
2,Y477110119_lag2,Y477110119_lag1,1.0000,duplicate column
3,Y477110116_lag2,Y477110116_lag1,1.0000,duplicate column
4,Y477110121_lag2,Y477110121_lag1,1.0000,duplicate column
5,Y477110038_lag2,Y477110038_lag1,1.0000,duplicate column
6,Y477110037_lag2,Y477110037_lag1,1.0000,duplicate column
7,Y477110036_lag2,Y477110036_lag1,1.0000,duplicate column
8,Y477110035_lag2,Y477110035_lag1,1.0000,duplicate column
9,Y477100023_lag2,Y477100023_lag1,0.9997,high_corr |r|=1.000; kept by higher importance...


In [23]:
# Важность признаков (gain + permutation)
imp.head(15)

,feature,gain,perm,score
0,nom_log_lag1,0.795463,0.896010,0.845737
1,real_log_lag1,0.043718,0.065765,0.054742
2,year_norm,0.075612,0.000000,0.037806
3,Y477110378_lag1,0.031994,0.005463,0.018729
4,Y477110108_lag1,0.011552,0.017579,0.014566
5,cluster_id,0.013155,0.000000,0.006577
6,Y477110341_lag2,0.000905,0.005235,0.003070
7,fo_Северо-Кавказский ФО,0.003109,0.001227,0.002168
8,Y477100014_lag1,0.000206,0.003571,0.001889
9,gdp_growth_rf_wb_lag1,0.002898,0.000000,0.001449


In [24]:
# VIF-диагностика
vif = pd.read_csv(RESULTS / 'vif_check.csv')
vif.head(20)

,feature,vif
0,key_rate_avg_lag1,1000000.000000
1,usd_rub_avg_lag1,1000000.000000
2,gdp_growth_rf_wb_lag1,1000000.000000
3,inflation_rf_wb_lag1,1000000.000000
4,unemployment_rf_wb_lag1,1000000.000000
5,industrial_production_rf_rosstat_lag1,1000000.000000
6,oil_brent_price_avg_usd_lag1,1000000.000000
7,year_norm,1000000.000000
8,nom_log_lag1,161.877180
9,real_log_lag1,145.033294


## 8. Корреляционный анализ


In [25]:
# Идентичная heatmap, что и в Streamlit-app (модуль viz_common)
vc.fig_correlation_heatmap(df, sel, max_feat=18).show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Корреляции с целевой переменной (тот же расчёт, что в Streamlit)
import plotly.express as px
train = df[df['year'] <= TRAIN_END][sel + ['nom_log', 'real_log']].dropna()
nom_c = train[sel].corrwith(train['nom_log']).sort_values()
real_c = train[sel].corrwith(train['real_log']).sort_values()

fig = px.bar(nom_c.tail(15).iloc[::-1], orientation='h',
              template='plotly_white', color_discrete_sequence=[vc.COLOR_NOM],
              title='Топ-15 ρ с ном. log(ВРП/чел.)')
fig.update_layout(yaxis_title='', showlegend=False, height=480)
fig.show()
fig = px.bar(real_c.tail(15).iloc[::-1], orientation='h',
              template='plotly_white', color_discrete_sequence=[vc.COLOR_REAL],
              title='Топ-15 ρ с реал. log(ВРП/чел.)')
fig.update_layout(yaxis_title='', showlegend=False, height=480)
fig.show()

## 9. Сравнение классических и современных методов

### Группы моделей:

**Базовые бенчмарки:**
* **Naive**: $\hat y_t = y_{t-1}$.
* **MeanGrowth**: $\hat y_t = y_{t-1} \cdot (1 + \bar g_{region})$.

**Эконометрика / регуляризованные:**
* **Linear / Ridge / Lasso / ElasticNet** на стандартизованных признаках.

**Деревья и ансамбли:**
* **RandomForest** (баггинг).
* **GradientBoosting** (sklearn).
* **XGBoost**.
* **LightGBM**.
* **CatBoost**.

**Нейросеть:**
* **MLPRegressor** (2 скрытых слоя, tanh, early stopping).

Все ML обучены на `log(target)`, прогноз обратно через `expm1`.

### 9.1. Walk-forward CV
Для каждого test_year ∈ [2020; 2023]:
* train = year ≤ test_year - 1
* test  = year == test_year

Это **честный out-of-sample** в условиях панельных данных, включая COVID и санкции.


In [ ]:
summary = pd.read_csv(RESULTS / 'model_summary.csv', index_col='model')
summary.round(4)

In [ ]:
# Графики из единого модуля viz_common (тот же стиль, что в Streamlit)
vc.fig_model_comparison(summary, metric='RMSLE').show()
vc.fig_model_comparison(summary, metric='MAPE').show()

In [ ]:
# Динамика по годам (та же фигура, что в Streamlit на вкладке «Сравнение моделей»)
py = pd.read_csv(RESULTS / 'per_year_metrics.csv')
py_n = py[py['target'] == 'nominal'] if 'target' in py.columns else py
vc.fig_mape_by_year(py_n).show()

### 9.2. Сравнение для номинального и реального ВРП

Раздельные walk-forward на обоих таргетах.


In [ ]:
# Метрики для номинального и реального
sum_nom = pd.read_csv(RESULTS / 'model_summary_nominal.csv', index_col='model')
sum_real = pd.read_csv(RESULTS / 'model_summary_real.csv', index_col='model')

cmp = pd.DataFrame({
    'MAPE_nom': sum_nom['MAPE'],
    'MAPE_real': sum_real['MAPE'],
    'RMSLE_nom': sum_nom['RMSLE'],
    'RMSLE_real': sum_real['RMSLE'],
}).sort_values('MAPE_nom')
cmp.round(3)

## 12. Поиск гиперпараметров (TimeSeriesSplit)

Подбор по сетке для:
* Ridge / ElasticNet;
* RandomForest;
* GradientBoosting;
* XGBoost;
* LightGBM;
* CatBoost (если установлен).

Метод: **TimeSeriesSplit ПО ГОДАМ** внутри тренировочного периода
(year ≤ 2019). Тестовый период (2020–2023) полностью изолирован.


In [ ]:
gs = pd.read_csv(RESULTS / 'grid_search_results.csv')
gs

## 13. Выбор лучшей модели для сценарного прогноза

### Критерии выбора (комплексные):
1. **Качество на walk-forward**: RMSLE, MAPE.
2. **Устойчивость во времени**: MAPE-разброс по тестовым годам.
3. **Поддержка нелинейностей**: важна для сценарного what-if.
4. **Интерпретируемость**: возможность SHAP-разложения.
5. **Скорость**: модель должна обновляться при появлении новых данных.

### Вывод:
* **MeanGrowth** — лучший по MAPE, но не поддерживает what-if.
* **RandomForest / GradientBoosting / LightGBM** — лучшие ML, поддерживают SHAP.
* **GradientBoosting** выбран для сценарного прогноза: стабильный базовый
  тренд + работает с подменой макрофакторов.
* **LightGBM** — для SHAP-интерпретации.

См. файл `models/gradientboosting.pkl`.


## 14. Feature Importance


In [ ]:
# gain + permutation (на той же diff-шкале, что показывает Streamlit)
import plotly.express as px
imp_top = imp.head(15).iloc[::-1]
long = imp_top.melt(id_vars='feature', value_vars=['gain', 'perm'],
                     var_name='тип', value_name='важность')
fig = px.bar(long, x='важность', y='feature', color='тип',
              orientation='h', barmode='group',
              template='plotly_white', height=560,
              title='TOP-15: gain vs permutation importance')
fig.update_layout(yaxis_title='')
fig.show()

## 15. SHAP-анализ

Модель: LightGBM, обученный на train ≤ 2019; SHAP TreeExplainer на test 2020–2023.

* **shap_global** — средние |SHAP| по всем регионам/годам;
* **shap_per_cluster** — по типам регионов (4 кластера);
* **shap_per_region** — по конкретным регионам.


In [ ]:
# SHAP-1: модель УРОВНЯ (доминирует lag1 — структурное свойство автокорреляции)
shap_g = pd.read_csv(RESULTS / 'shap_global.csv')
vc.fig_shap_global(shap_g, top_n=15,
    title='Level-модель: важность признаков (LightGBM)').show()

In [ ]:
# SHAP по типам регионов
shap_c = pd.read_csv(RESULTS / 'shap_per_cluster.csv')
vc.fig_shap_per_cluster(shap_c, shap_g, top_n=10).show()

## 16. Прогноз на 1–5 лет

Рекурсивный прогноз: для каждого региона последовательно прогнозируется
2024, 2025, ..., 2028. На каждом шаге используется лаг предыдущего прогноза.

Для каждого региона и горизонта строятся прогнозы по **3 сценариям**:
* **baseline** — текущие тенденции (умеренный рост);
* **optimistic** — рост инвестиций, низкая инфляция, высокая нефть;
* **pessimistic** — стагнация, высокая инфляция, низкая нефть.


In [ ]:
fc = pd.read_csv(RESULTS / 'forecast_all_scenarios.csv')
print('Shape:', fc.shape)
fc.head(12)

In [ ]:
# Сценарный прогноз для Москвы — идентичный график, что в Streamlit
region = 'Москва' if 'Москва' in df['object_name'].values else 'г. Москва'
vc.fig_scenario_forecast(df, fc, region, target='nominal').show()
vc.fig_scenario_forecast(df, fc, region, target='real').show()

## 17. Сценарные предпосылки

| Параметр | Базовый | Оптимистичный | Пессимистичный |
|----------|---------|---------------|----------------|
| Нефть Brent, $/барр. | 75 | 95 | 60 |
| Ключевая ставка, % | 12 | 8 | 16 |
| Инфляция, % | 6.5 | 4.5 | 9.5 |
| Курс USD/RUB | 90 | 80 | 100 |
| Рост ВВП РФ, % | 2.5 | 3.5 | 0.8 |
| Безработица РФ, % | 3.5 | 3.0 | 5.0 |
| Промышленность РФ, % | +2.5 | +3.5 | -0.5 |

**Дополнительный канал с 2025 года**:
* номинальный коэффициент: opt +1%/год, baseline 0, pess -1%/год;
* реальный коэффициент: opt +1%/год, baseline 0, pess -1.2%/год.

Первый прогнозный год не получает ручной сценарной надбавки: переход 2023→2024
строится от фактического уровня 2023 года, чтобы не создавать искусственный скачок.

### Графики для нескольких регионов


In [ ]:
# Прогнозы для нескольких регионов (одинаковые с Streamlit)
for reg in ['Москва', 'Ханты-Мансийский автономный округ - Югра',
             'Республика Татарстан', 'Республика Дагестан']:
    if reg in df['object_name'].values:
        vc.fig_scenario_forecast(df, fc, reg, target='nominal').show()

## 18. Карта РФ с границами регионов

Интерактивные карты с раскраской регионов по значению ВРП:
* **Факт** за последний год — номинальный и реальный;
* **Прогноз** по трём сценариям на горизонте 1, 3, 5 лет.

При отсутствии GeoJSON используется bubble-map (Plotly Scattermapbox)
на основе центроидов регионов (`region_centroids.json`).

Файлы сохранены в `reports/maps/*.html`.


In [ ]:
# Choropleth-карта России — те же границы регионов, что в Streamlit
last_year = df['year'].max()
last = df[df['year']==last_year][['object_name', TARGET_NOM, GRP_REAL_PC]].copy()
vc.fig_choropleth(last, value_col=TARGET_NOM,
                   title=f'Номинальный ВРП/чел., {last_year}',
                   colorscale='Blues').show()
vc.fig_choropleth(last, value_col=GRP_REAL_PC,
                   title=f'Реальный ВРП/чел. в ценах {BASE_YEAR}, {last_year}',
                   colorscale='Oranges').show()

In [ ]:
# Карта прогноза на h=5 (2028) по сценариям
for sc in ['baseline', 'optimistic', 'pessimistic']:
    sub = fc[(fc['scenario']==sc) & (fc['horizon']==5)]
    cscale = {'baseline':'Blues','optimistic':'Greens','pessimistic':'Reds'}[sc]
    vc.fig_choropleth(sub, value_col='y_pred_nominal',
                       title=f'Прогноз ном. ВРП/чел. ({sc}, 2028)',
                       colorscale=cscale).show()

## 19. Интерфейс аналитической системы

Реализован как **Streamlit-приложение** в `app_vkr/streamlit_app.py`.

### 7 разделов:
1. **Главная** — KPI, динамика медианного ВРП.
2. **Анализ данных** — распределения, корреляции, кластеры.
3. **Прогноз региона** — выбор региона + горизонт + сценарий, SHAP-вклад.
4. **Сценарный анализ** — распределения и разрыв сценариев.
5. **Карта регионов** — интерактивная с фильтрами.
6. **Интерпретация модели (SHAP)** — глобально/по кластерам.
7. **Методология** — описание подхода.

### Запуск:
```bash
streamlit run app_vkr/streamlit_app.py
```


## 20. Практическая ценность

### Кому полезна разработанная система:

| Группа | Применение |
|---------|------------|
| **Минэкономразвития РФ** | мониторинг региональной динамики, краткосрочный прогноз |
| **Минфин РФ** | бюджетное планирование, оценка налоговой базы регионов |
| **Региональные правительства** | анализ собственного прогноза, бенчмаркинг |
| **Аналитические центры** | сценарный анализ, оценка эффектов реформ |
| **Инвесторы** | оценка инвестиционной привлекательности территорий |
| **Учёные** | панельные регрессии, эконометрические исследования |
| **Преподаватели вузов** | учебный кейс по ML и эконометрике |

### Конкретные сценарии использования:
* **Раннее предупреждение** регионов риска (стресс-тест на пессимистичный сценарий);
* **Оценка эффекта** изменения макрополитики (нефть, ставка, курс);
* **Бенчмаркинг** регионов внутри кластеров;
* **Поддержка решений** по распределению трансфертов;
* **Мониторинг неравенства** (Джини, top-10/bottom-10 спред).


## 21. Ограничения исследования

1. **Неполнота статистических данных:** часть индикаторов имеет покрытие < 90%,
   обработка SimpleImputer(median) может вносить смещение.
2. **Задержка публикации ВРП:** ~1.5–2 года (например, 2023 опубликован в 2025).
3. **Изменения методик Росстата** (например, ОКВЭД2 с 2017) — структурные сдвиги.
4. **Внешние шоки** (COVID-2020, санкции-2022) — режимные изменения, плохо
   улавливаются короткими рядами.
5. **Не учтены** новые регионы 2022 г. (ЛНР, ДНР, Запорожская, Херсонская
   области) — нет полных рядов.
6. **Сценарии** — упрощённое описание; реальная макрополитика сложнее.
7. **Доминирование AR-компоненты:** SHAP `log(ВРП)_lag1` ≈ 73% всех вкладов,
   что ограничивает информативность остальных факторов.
8. **Демографические и институциональные факторы** (качество институтов, доверие,
   социальный капитал) не измеряются Росстатом — игнорируются.
9. **Эффекты пространственной корреляции** между соседними регионами
   (spatial spillovers) не учтены.
10. **Риск переобучения**: 17 лет × 85 регионов = ~1445 наблюдений, что
    маловато для глубоких MLP.


## 22. Итоговые выводы

1. **Современные ML-методы** (RandomForest, GradientBoosting, LightGBM, XGBoost)
   значимо превосходят регуляризованные линейные модели (Ridge/Lasso/ElasticNet)
   на walk-forward CV 2020–2023:
   * GradientBoosting/LightGBM MAPE ≈ 9.5–10%;
   * Ridge/Lasso/ElasticNet MAPE ≈ 17–18%.

2. **Инерционный бенчмарк MeanGrowth** (региональная экстраполяция среднего
   роста) оказывается **сильным конкурентом** на 1-летнем горизонте
   (MAPE ≈ 8.7%) — типичная картина для коротких макроэкономических рядов.
   Это **подтверждает гипотезу**, что ML особенно полезен для сценарного
   what-if и мультивариативного анализа, а не для чистого экстраполирования.

3. **Лаги целевой переменной** (target_lag1) — доминирующий предиктор
   (SHAP ≈ 73%). Это структурное свойство ряда с высокой автокорреляцией.

4. **Ключевые экзогенные факторы:**
   * макроусловия (ключевая ставка, курс USD/RUB, нефть);
   * среднемесячная зарплата (производительность труда);
   * инвестиции в основной капитал на душу;
   * розничный товарооборот.

5. **Регионы-лидеры по прогнозу:** Москва, ХМАО, ЯНАО, Татарстан,
   Санкт-Петербург — стабильно высокий уровень ВРП/чел. сохраняется.

6. **Регионы риска** в пессимистичном сценарии: СКФО (Дагестан, Ингушетия,
   Чечня), Тыва, Алтай — высокая чувствительность к снижению трансфертов
   и нефтегазовой ренты.

7. **Сценарный разрыв** (опт − пес) на h=5 лет составляет ≈ 35–45% от
   базового прогноза, что соответствует историческим амплитудам кризисов
   (2008, 2014, 2020, 2022).

8. **Аналитическая система** реализована как Streamlit-приложение и
   готова к использованию в государственном секторе и аналитических центрах.

### Направления развития
* Добавление пространственной корреляции (spatial econometrics);
* Учёт ретроспективных пересмотров (vintage data);
* Прогноз компонент ВРП (по ВЭД);
* Ансамблирование ML + эконометрических моделей;
* Включение нестандартных данных (Яндекс.ДОМ, мобильные операторы, ОФД).
